# Fase 4 · KPIs y datos para el dashboard
**Responsabilidad:** calcular los 6 indicadores clave y preparar las tablas que consume Tableau.
No vuelve a limpiar el archivo original ni cambia la población analítica (`incluido_analisis = True`).

| | |
|---|---|
| **Entrada** | `salida_fase3/airbnb_bogota_analitico.csv`, `salida_fase3/anfitriones_resumen.csv` |
| **Salida** | `resultados/kpis.csv`, `resultados/anuncios_dashboard.csv`, `resultados/anfitriones_dashboard.csv` |
| **Anterior / siguiente** | `01_preparacion_datos` → `03_contrastes_hipotesis` |

## 1. Carga

In [1]:
import numpy as np
import pandas as pd

from bi_airbnb import DIR_RESULTADOS, anfitriones_unicos, cargar_base, guardar_csv

todo, d, hosts = cargar_base()
print(f"Anuncios analizados: {len(d):,} de {len(todo):,} | anfitriones: {len(hosts):,}")

Anuncios analizados: 17,998 de 19,505 | anfitriones: 7,795


## 2. KPIs
Cada fila lleva su `n_validos`, la hipótesis que alimenta y la fórmula equivalente en Tableau.

In [2]:
host_count = anfitriones_unicos(d)
rated = d.calificacion.notna().sum()

kpis = pd.DataFrame([
    ["K1", "Precio mediano por noche", d.precio_cop.median(), "COP", len(d), "H1 y H2", "MEDIAN([precio_cop])"],
    ["K2", "Precio mediano por huésped", d.precio_por_huesped.median(), "COP", d.precio_por_huesped.count(), "H3", "MEDIAN([precio_por_huesped])"],
    ["K3", "Calificación promedio", d.calificacion.mean(), "puntos sobre 5", rated, "H4 y H5", "AVG([calificacion])"],
    ["K4", "Anuncios con reseñas en 12 meses", d.resenas_12m.gt(0).mean(), "fracción", d.resenas_12m.count(), "H5", "SUM(IIF([resenas_12m]>0,1,0))/COUNT([resenas_12m])"],
    ["K5", "Ocupación estimada mediana", d.ocupacion_estimada_pct.median(), "fracción", d.ocupacion_estimada_pct.count(), "H5", "MEDIAN([ocupacion_estimada_pct])"],
    ["K6", "Anfitriones con varios anuncios", host_count.n_anuncios.ge(2).mean(), "fracción", len(host_count), "H5", "COUNTD(IF [n_anuncios]>=2 THEN [id_host] END)/COUNTD([id_host])"],
], columns=["id", "indicador", "valor", "unidad", "n_validos", "hipotesis", "formula_tableau"])
guardar_csv(kpis, DIR_RESULTADOS, "kpis.csv")
kpis[["id", "indicador", "valor", "n_validos"]]

,id,indicador,valor,n_validos
0,K1,Precio mediano por noche,160000.000000,17998
1,K2,Precio mediano por huésped,60000.000000,17998
2,K3,Calificación promedio,4.704137,14549
3,K4,Anuncios con reseñas en 12 meses,0.671797,17998
4,K5,Ocupación estimada mediana,0.065753,17998
5,K6,Anfitriones con varios anuncios,0.305067,7795


## 3. Tablas para Tableau
Los prefijos `a_` y `h_` garantizan que Tableau importe las claves como **texto**.

In [3]:
anuncios = d.copy()
anuncios["id_anuncio"] = "a_" + anuncios["id_anuncio"]
anuncios["id_host"] = "h_" + anuncios["id_host"]
anuncios["precio_intervalo_50mil"] = (np.floor(anuncios["precio_cop"] / 50000) * 50000).astype(int)
anuncios["activo_12m"] = anuncios["resenas_12m"].gt(0).astype(int)
anuncios["grupo_anfitrion"] = np.where(anuncios["n_anuncios"] >= 2, "Varios anuncios", "Un anuncio")
guardar_csv(anuncios, DIR_RESULTADOS, "anuncios_dashboard.csv")

anfitriones = hosts.copy()
anfitriones["id_host"] = "h_" + anfitriones["id_host"]
guardar_csv(anfitriones, DIR_RESULTADOS, "anfitriones_dashboard.csv")
print(f"anuncios_dashboard: {anuncios.shape} | anfitriones_dashboard: {anfitriones.shape}")

anuncios_dashboard: (17998, 51) | anfitriones_dashboard: (7795, 9)
